# Reel Insights: An End-to-End Exploratory Data Analysis of the Global Movie Industry (TMDB)

##  Project Overview

This notebook focuses on performing SQL-based analysis of the TMDB movie dataset using SQLite.

The objective is to explore movie industry trends by querying relational data, analyzing movie performance, identifying popular genres, actors, directors, and keywords.

The analysis uses six cleaned datasets prepared during the data cleaning stage.

##  SQL Analysis Workflow

- Create an SQLite database.
- Load the six cleaned CSV datasets.
- Create relational tables in SQLite.
- Verify database tables and records.
- Test relationships between tables using SQL JOIN operations.
- Solve 10 SQL analysis questions to extract meaningful insights.

##  Import Required Libraries

The following Python libraries are used for database creation, data loading, and SQL query execution.

- **Pandas:** Used to load CSV files and retrieve SQL query results as DataFrames.
- **SQLite3:** Used to create and connect to the SQLite database.

In [1]:
import pandas as pd
import sqlite3

##  Create SQLite Database

An SQLite database named `tmdb_movies.db` is created to store the six cleaned TMDB datasets.

The `sqlite3.connect()` function establishes a connection to the database. If the database file does not exist, SQLite creates it automatically.

In [2]:
conn = sqlite3.connect("tmdb_movies.db")

In [3]:
print(conn)

##  Load Cleaned Datasets

The six cleaned CSV files generated during the data cleaning stage are loaded into Pandas DataFrames.

These datasets will be transferred into the SQLite database as individual tables.

The original CSV files are preserved separately in the `data` folder.

In [10]:
movies_df = pd.read_csv("cleaned_data/movies.csv")


In [8]:
genres_df = pd.read_csv("cleaned_data/genres.csv")
movie_genres_df = pd.read_csv("cleaned_data/movie_genres.csv")
cast_df = pd.read_csv("cleaned_data/cast.csv")
crew_df = pd.read_csv("cleaned_data/crew.csv")
movie_keywords_df = pd.read_csv("cleaned_data/movie_keywords.csv")

In [9]:
print(movies_df.shape)
print(genres_df.shape)
print(movie_genres_df.shape)
print(cast_df.shape)
print(crew_df.shape)
print(movie_keywords_df.shape)

(2503, 12)
(19, 2)
(6878, 2)
(24902, 6)
(7289, 6)
(41681, 3)


##  Create SQL Tables

The six cleaned DataFrames are imported into the SQLite database using the Pandas `to_sql()` method.

Each DataFrame is stored as a separate SQL table. The `if_exists='replace'` option replaces an existing table, while `index=False` prevents the DataFrame index from being stored as an additional column.

In [11]:
movies_df.to_sql("movies", conn, if_exists="replace", index=False)
genres_df.to_sql("genres", conn, if_exists="replace", index=False)
movie_genres_df.to_sql("movie_genres", conn, if_exists="replace", index=False)
cast_df.to_sql("cast", conn, if_exists="replace", index=False)
crew_df.to_sql("crew", conn, if_exists="replace", index=False)
movie_keywords_df.to_sql("movie_keywords", conn, if_exists="replace", index=False)


41681

In [12]:
pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,movies
1,genres
2,movie_genres
3,cast
4,crew
5,movie_keywords


In [13]:
query = """
SELECT *
FROM movies
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...
1,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es..."
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...
3,155,The Dark Knight,en,2008-07-16,152,185000000.0,1.004558e+09,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...
4,19995,Avatar,en,2009-12-16,162,237000000.0,2.923706e+09,48.1996,7.610,34348,Released,"In the 22nd century, a paraplegic Marine is di..."


In [15]:
query = """
SELECT COUNT(*)
FROM movies;
"""

pd.read_sql_query(query, conn)

,COUNT(*)
0,2503


In [16]:
query = """
SELECT
    m.title,
    g.genre_name
FROM movies AS m
INNER JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
INNER JOIN genres AS g
    ON mg.genre_id = g.genre_id
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,title,genre_name
0,Interstellar,Adventure
1,Interstellar,Drama
2,Interstellar,Science Fiction
3,Inception,Adventure
4,Inception,Action
5,Inception,Science Fiction
6,The Avengers,Adventure
7,The Avengers,Action
8,The Avengers,Science Fiction
9,The Dark Knight,Action


## Test SQL JOIN Operations

An INNER JOIN is performed across the movies, movie_genres, and genres tables to verify the relationships between the tables.

The movie_genres table acts as a bridge between movies and genres.

The query retrieves movie titles along with their corresponding genre names.

In [17]:
query = """
SELECT
    m.title,
    g.genre_name
FROM movies AS m
INNER JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
INNER JOIN genres AS g
    ON mg.genre_id = g.genre_id
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,title,genre_name
0,Interstellar,Adventure
1,Interstellar,Drama
2,Interstellar,Science Fiction
3,Inception,Adventure
4,Inception,Action
5,Inception,Science Fiction
6,The Avengers,Adventure
7,The Avengers,Action
8,The Avengers,Science Fiction
9,The Dark Knight,Action


### Problem 1: Find all Action movies released after 2015.

In [18]:
query = """
SELECT
    m.movie_id,
    m.title,
    m.release_date,
    g.genre_name
FROM movies AS m
INNER JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
INNER JOIN genres AS g
    ON mg.genre_id = g.genre_id
WHERE
    g.genre_name = 'Action'
    AND CAST(strftime('%Y', m.release_date) AS INTEGER) > 2015
ORDER BY m.release_date;
"""

pd.read_sql_query(query, conn)

,movie_id,title,release_date,genre_name
0,299687,The 5th Wave,2016-01-14,Action
1,140300,Kung Fu Panda 3,2016-01-23,Action
2,329833,Zoolander 2,2016-02-06,Action
3,293660,Deadpool,2016-02-09,Action
4,146198,Triple 9,2016-02-19,Action
...,...,...,...,...
263,911430,F1,2025-06-25,Action
264,1061474,Superman,2025-07-09,Action
265,617126,The Fantastic 4: First Steps,2025-07-23,Action
266,1242898,Predator: Badlands,2025-11-05,Action


**Objective:** Identify all Action genre movies released after 2015.

**Approach:**
- Joined the `movies`, `movie_genres`, and `genres` tables using their matching IDs.
- Filtered records where the genre name is 'Action'.
- Used SQLite's `strftime()` function to extract the release year.
- Selected movies released after 2015.
- Sorted the results by release date.

**Observation:** The query returns Action movies released from 2016 onwards.

### Problem 2 – Top 10 Highest-Grossing Movies by Genre



In [19]:
query = """
SELECT
    m.movie_id,
    m.title,
    m.revenue,
    g.genre_name
FROM movies AS m
INNER JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
INNER JOIN genres AS g
    ON mg.genre_id = g.genre_id
ORDER BY m.revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,movie_id,title,revenue,genre_name
0,19995,Avatar,2.923706e+09,Adventure
1,19995,Avatar,2.923706e+09,Action
2,19995,Avatar,2.923706e+09,Science Fiction
3,299534,Avengers: Endgame,2.799439e+09,Adventure
4,299534,Avengers: Endgame,2.799439e+09,Action
5,299534,Avengers: Endgame,2.799439e+09,Science Fiction
6,76600,Avatar: The Way of Water,2.334485e+09,Adventure
7,76600,Avatar: The Way of Water,2.334485e+09,Action
8,76600,Avatar: The Way of Water,2.334485e+09,Science Fiction
9,597,Titanic,2.264162e+09,Drama


**Approach:**
- Joined the `movies`, `movie_genres`, and `genres` tables.
- Selected movie titles, revenue, and genre names.
- Sorted the results by revenue in descending order.
- Used LIMIT 10 to retrieve the top 10 records.

**Observation:** 

Avatar has the highest revenue in the dataset, approximately $2.92 billion.

Avengers: Endgame ranks next, with approximately $2.80 billion.

Avatar: The Way of Water follows with approximately $2.33 billion.

Movies with multiple genres appear in multiple rows because of the many-to-many relationship between movies and genres.

Conclusion: The query successfully identifies the highest-grossing movie-genre combinations in the dataset.

### Problem 3 -Calculate the average budget and average revenue for each genre.

In [20]:
query = """
SELECT
    g.genre_name,
    AVG(m.budget) AS average_budget,
    AVG(m.revenue) AS average_revenue
FROM movies AS m
INNER JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
INNER JOIN genres AS g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
ORDER BY average_revenue DESC;
"""

pd.read_sql_query(query, conn)

,genre_name,average_budget,average_revenue
0,Adventure,1.098347e+08,3.936523e+08
1,Animation,8.531182e+07,3.641760e+08
2,Family,8.609588e+07,3.470590e+08
3,Science Fiction,9.586467e+07,3.276815e+08
4,Action,9.369061e+07,3.037658e+08
5,Fantasy,8.728895e+07,2.980836e+08
6,Comedy,5.398639e+07,2.176118e+08
7,Music,4.270204e+07,2.002342e+08
8,Romance,3.713262e+07,1.721627e+08
9,Thriller,5.081414e+07,1.665910e+08


**Observations:**
- Adventure has the highest average revenue, approximately $393.65 million.
- Animation ranks second with approximately $364.18 million in average revenue.
- Family ranks third with approximately $347.06 million.
- Adventure also has the highest average budget, approximately $109.83 million.
- Horror has the lowest average budget, approximately $26.36 million.
- TV Movie has the lowest average revenue, approximately $102.23 million.

**Conclusion:** Adventure, Animation, and Family are the top three genres based on average revenue in this dataset.

**Note:** Budget and revenue values include median-imputed records from the data cleaning stage. These averages should therefore be interpreted with caution.

### Problem 4 - Find the top 10 actors who have appeared in the highest number of movies.

In [21]:
query = """
SELECT
    actor_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM cast
GROUP BY person_id, actor_name
ORDER BY movie_count DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Brad Pitt,38
2,Robert De Niro,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Scarlett Johansson,32
6,Willem Dafoe,32
7,Mark Wahlberg,32
8,Morgan Freeman,31
9,Tom Cruise,31


**Observations:**
- Samuel L. Jackson ranks first with 39 movie appearances.
- Brad Pitt and Robert De Niro share second place with 38 movies each.
- Johnny Depp ranks next with 37 movie appearances.
- Tom Hanks appears in 33 movies.
- Scarlett Johansson, Willem Dafoe, and Mark Wahlberg each appear in 32 movies.

**Conclusion:** Samuel L. Jackson has the highest number of movie appearances among the actors in this dataset.

### Problem 5 - Find directors who have directed more than 3 movies.

In [22]:
query = """
SELECT
    person_id,
    person_name AS director_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE job = 'Director'
GROUP BY person_id, person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC;
"""

pd.read_sql_query(query, conn)

,person_id,director_name,movie_count
0,488,Steven Spielberg,27
1,510,Tim Burton,19
2,578,Ridley Scott,19
3,1032,Martin Scorsese,16
4,24,Robert Zemeckis,15
...,...,...,...
212,138781,Robert Eggers,4
213,1056121,Ryan Coogler,4
214,1113116,Andy Muschietti,4
215,1179066,Wes Ball,4


**Observations:**
- A total of 217 directors have directed more than three movies.
- Steven Spielberg ranks first with 27 movies.
- Tim Burton and Ridley Scott share the second position with 19 movies each.
- Martin Scorsese has directed 16 movies.
- Robert Zemeckis has directed 15 movies.

**Conclusion:** Steven Spielberg has the highest number of directing credits among the directors identified in the dataset.

### Problem 6 - Find the top 10 most frequently used movie keywords.

In [23]:
query = """
SELECT
    keyword_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords
GROUP BY keyword_id, keyword_name
ORDER BY movie_count DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


**Observations:**
- "Based on novel or book" is the most frequently used keyword, appearing in 449 movies.
- "Sequel" ranks second with 416 movies.
- "Duringcreditsstinger" appears in 296 movies.
- "Aftercreditsstinger" appears in 242 movies.
- "Murder" appears in 176 movies.
- Keywords related to adaptations, sequels, and post-credit scenes are prominent in the dataset.

**Conclusion:** "Based on novel or book" is the most frequently associated keyword among the movies in the dataset.

### Problem 7 - Find movies with a budget higher than the average budget but revenue lower than the average revenue.

In [24]:
query = """
SELECT
    movie_id,
    title,
    budget,
    revenue
FROM movies
WHERE budget > (SELECT AVG(budget) FROM movies)
AND revenue < (SELECT AVG(revenue) FROM movies)
ORDER BY budget DESC;
"""

pd.read_sql_query(query, conn)

,movie_id,title,budget,revenue
0,609681,The Marvels,274800000.0,206136825.0
1,845781,Red One,250000000.0,185700759.0
2,812583,Wake Up Dead Man: A Knives Out Mystery,210000000.0,4000000.0
3,464052,Wonder Woman 1984,200000000.0,169601036.0
4,508943,Luca,200000000.0,51074773.0
...,...,...,...,...
274,1370,Rambo III,63000000.0,189015611.0
275,345887,The Equalizer 2,62000000.0,190400157.0
276,8202,Æon Flux,62000000.0,53321673.0
277,7485,Shooter,61000000.0,95700000.0


**Observations:**
- A total of 279 movies have budgets above the overall average and revenues below the overall average.
- The Marvels has the highest budget among the displayed results, at approximately $274.8 million.
- Red One follows with a budget of $250 million.
- Wake Up Dead Man has a budget of $210 million but recorded only $4 million in revenue.

**Conclusion:** The query identifies movies with relatively high production budgets but comparatively low revenue.

**Note:** Budget and revenue values include median-imputed records. Therefore, the results should be interpreted with caution.

### Problem 8 - Find all actors who have acted in Christopher Nolan movies.

In [25]:
query = """
SELECT DISTINCT
    c.person_id,
    c.actor_name
FROM cast AS c
INNER JOIN crew AS cr
    ON c.movie_id = cr.movie_id
WHERE cr.person_name = 'Christopher Nolan'
AND cr.job = 'Director'
ORDER BY c.actor_name;
"""

pd.read_sql_query(query, conn)

,person_id,actor_name
0,6383,Aaron Eckhart
1,1158,Al Pacino
2,16607,Alon Aboutboul
3,1333,Andy Serkis
4,145143,Aneurin Barnard
...,...,...
79,2524,Tom Hardy
80,207,Tom Wilkinson
81,17052,Topher Grace
82,8210,Wes Bentley


**Observations:**
- The query identified 84 unique actors who appeared in movies directed by Christopher Nolan.
- Aaron Eckhart, Al Pacino, Andy Serkis, Tom Hardy, and other actors were included in the results.
- Actors appearing in multiple Christopher Nolan movies were listed only once using DISTINCT.

**Conclusion:** The query successfully identifies the unique cast members associated with Christopher Nolan's directing credits in the dataset.

### Problem 9 - Find the genre with the highest average rating, considering only genres with at least 20 movies.

In [26]:
query = """
SELECT
    g.genre_name,
    AVG(m.vote_average) AS average_rating,
    COUNT(DISTINCT m.movie_id) AS movie_count
FROM movies AS m
INNER JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
INNER JOIN genres AS g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY average_rating DESC
LIMIT 1;
"""

pd.read_sql_query(query, conn)

,genre_name,average_rating,movie_count
0,War,7.357576,85


**Objective:**
Find the genre with the highest average movie rating, considering only genres with at least 20 movies.

**Approach:**
- Joined the movies, movie_genres, and genres tables.
- Used AVG() to calculate the average movie rating for each genre.
- Used COUNT(DISTINCT movie_id) to count unique movies in each genre.
- Applied HAVING to include only genres with at least 20 movies.
- Sorted the results by average rating in descending order and selected the top genre.

**Observation:**
- War is the highest average-rated genre.
- Its average rating is approximately 7.36.
- The War genre contains 85 movies in the dataset.

**Conclusion:**
Among genres with at least 20 movies, War has the highest average TMDB rating.

### Problem 10: Find the Top 10 Movies with the Highest Number of Keywords

In [27]:
query = """
SELECT
    m.movie_id,
    m.title,
    COUNT(DISTINCT mk.keyword_id) AS keyword_count
FROM movies AS m
INNER JOIN movie_keywords AS mk
    ON m.movie_id = mk.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,movie_id,title,keyword_count
0,1100782,Smile 2,98
1,260346,Taken 3,70
2,588,Silent Hill,63
3,63,Twelve Monkeys,57
4,281338,War for the Planet of the Apes,54
5,694,The Shining,52
6,119450,Dawn of the Planet of the Apes,51
7,10191,How to Train Your Dragon,50
8,166428,How to Train Your Dragon: The Hidden World,49
9,268896,Pacific Rim: Uprising,49


**Objective:**
Identify the top 10 movies associated with the highest number of unique keywords.

**Approach:**
- Joined the movies and movie_keywords tables using movie_id.
- Used COUNT(DISTINCT keyword_id) to calculate the number of unique keywords for each movie.
- Grouped the results by movie_id and title.
- Sorted the movies in descending order of keyword count.
- Used LIMIT 10 to retrieve the top 10 movies.

**Observation:**
- Smile 2 has the highest keyword count, with 98 keywords.
- Taken 3 ranks second with 70 keywords.
- Silent Hill ranks third with 63 keywords.
- Two Planet of the Apes movies and two How to Train Your Dragon movies appear in the top 10.

**Conclusion:**
Smile 2 has the highest number of associated keywords among the movies in the dataset, indicating that it has the most extensive keyword tagging in this collection.